[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/09_naive_bayes/02_gaussian_nb_from_scratch.ipynb)

# Bài 02 — Gaussian Naive Bayes from scratch

**🎯 Mục tiêu**
- Mô hình hoá feature liên tục: $P(x_j \mid y = c) = \mathcal N(x_j; \mu_{cj}, \sigma^2_{cj})$.
- Cài đặt Gaussian NB from scratch (fit = tính mean/var theo lớp!), tính trong không gian log.
- Thấy decision boundary **bậc hai**; vai trò prior; `var_smoothing`.
- Đối chiếu với `sklearn.naive_bayes.GaussianNB`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/09_naive_bayes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

from ml_course.visualization import PALETTE, plot_decision_boundary, set_style

set_style()
iris = pd.read_csv("../../data/sample/iris.csv")
iris.head()

## 1. Ý tưởng: mỗi lớp, mỗi feature một đường chuông

"Học" Gaussian NB chỉ là tính **trung bình và phương sai của từng feature trong từng lớp** — không có vòng lặp tối ưu nào! Với 3 lớp × 4 feature = 12 cặp $(\mu, \sigma^2)$ + 3 prior.

In [ ]:
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
for ax, f in zip(axes, features):
    grid = np.linspace(iris[f].min() - 0.5, iris[f].max() + 0.5, 300)
    for i, (species, g) in enumerate(iris.groupby("species")):
        ax.hist(g[f], bins=12, density=True, alpha=0.35, color=PALETTE[i])
        ax.plot(grid, norm.pdf(grid, g[f].mean(), g[f].std(ddof=0)), color=PALETTE[i], lw=2, label=species)
    ax.set_title(f)
axes[0].legend(fontsize=8)
plt.suptitle("Histogram (cột) và Gaussian ước lượng (đường) theo từng lớp", y=1.03)
plt.show()

`petal_length` và `petal_width` tách các lớp rất rõ — chỉ riêng chúng đã đủ phân loại tốt. Mỗi đường chuông là một $P(x_j \mid y = c)$.

## 2. Cài đặt from scratch

$$\log P(\mathbf x \mid y = c) = \sum_{j=1}^d \left[-\frac12\log(2\pi\sigma^2_{cj}) - \frac{(x_j - \mu_{cj})^2}{2\sigma^2_{cj}}\right]$$

In [ ]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9, priors=None):
        self.var_smoothing, self.priors = var_smoothing, priors

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        self.classes_ = np.unique(y)
        self.mu_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])              # (C, d)
        eps = self.var_smoothing * X.var(axis=0).max()                                   # tránh chia cho 0
        self.var_ = np.array([X[y == c].var(axis=0) for c in self.classes_]) + eps       # (C, d)
        prior = self.priors if self.priors is not None else [np.mean(y == c) for c in self.classes_]
        self.log_prior_ = np.log(np.asarray(prior, float))
        return self

    def joint_log_likelihood(self, X):
        X = np.asarray(X, float)[:, None, :]                                            # (n, 1, d)
        log_pdf = -0.5 * np.log(2 * np.pi * self.var_) - (X - self.mu_) ** 2 / (2 * self.var_)   # (n, C, d)
        return log_pdf.sum(axis=2) + self.log_prior_                                    # (n, C)

    def predict_proba(self, X):
        jll = self.joint_log_likelihood(X)
        return np.exp(jll - logsumexp(jll, axis=1, keepdims=True))

    def predict(self, X):
        return self.classes_[np.argmax(self.joint_log_likelihood(X), axis=1)]


X, y = iris[features].to_numpy(), iris["species"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
ours = GaussianNaiveBayes().fit(X_train, y_train)
sk = GaussianNB().fit(X_train, y_train)
print(f"accuracy scratch = {np.mean(ours.predict(X_test) == y_test):.4f} | sklearn = {sk.score(X_test, y_test):.4f}")
print("xác suất trùng sklearn:", np.allclose(ours.predict_proba(X_test), sk.predict_proba(X_test)))
pd.DataFrame(ours.mu_, index=ours.classes_, columns=features).round(2)

## 3. Decision boundary: bậc hai

Lấy log-tỉ số posterior của hai lớp: các số hạng $x_j^2 / (2\sigma^2_{cj})$ **không triệt tiêu** khi phương sai hai lớp khác nhau → ranh giới là đường **bậc hai** (parabol, ellipse, hyperbol). Nếu ép mọi lớp dùng chung phương sai, số hạng bậc hai triệt tiêu → ranh giới **tuyến tính**.

In [ ]:
rng = np.random.default_rng(1)
A = rng.normal([0, 0], [0.6, 0.6], size=(200, 2))           # lớp 0: tập trung
B = rng.normal([1.5, 0.5], [1.8, 1.4], size=(200, 2))       # lớp 1: trải rộng
X2 = np.vstack([A, B])
y2 = np.r_[np.zeros(200), np.ones(200)].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(GaussianNaiveBayes().fit(X2, y2), X2, y2, ax=axes[0])
axes[0].set_title("Gaussian NB: phương sai khác nhau → ranh giới cong (khép kín)")
from sklearn.linear_model import LogisticRegression

plot_decision_boundary(LogisticRegression().fit(X2, y2), X2, y2, ax=axes[1])
axes[1].set_title("Logistic Regression: ranh giới thẳng")
plt.show()

## 4. Prior thay đổi quyết định

Prior dịch chuyển ranh giới về phía lớp có prior nhỏ: với cùng $\mathbf x$, tăng $P(y = c)$ làm lớp $c$ dễ thắng hơn. Ví dụ: một điểm nằm giữa hai lớp `versicolor` và `virginica`:

In [ ]:
x_mid = np.array([[6.2, 2.9, 4.95, 1.65]])
for priors in [None, [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]]:
    m = GaussianNaiveBayes(priors=priors).fit(X_train, y_train)
    print(f"priors = {str(priors):16s} → {m.predict(x_mid)[0]:10s} | P = {m.predict_proba(x_mid).round(3)[0]}")

## 5. `var_smoothing` — khi một feature có phương sai (gần) 0 trong một lớp

Nếu trong tập train, một feature của một lớp luôn bằng cùng một giá trị, $\sigma^2_{cj} \approx 0$ → mọi điểm mới lệch giá trị đó **một chút** cũng có $\log P(x_j \mid y = c) \to -\infty$: lớp $c$ bị loại bất kể các feature khác. `var_smoothing` cộng thêm $\varepsilon \cdot \max_j \operatorname{Var}(x_j)$ vào mọi phương sai — vai trò giống Laplace smoothing cho dữ liệu liên tục. Mặc định của sklearn ($10^{-9}$) chỉ chống chia cho 0; muốn model "khoan dung" hơn cần $\varepsilon$ lớn hơn (chọn bằng CV).

In [ ]:
X_const = X_train.copy()
X_const[y_train == "setosa", 3] = 0.2                     # trong train, petal_width của setosa LUÔN = 0.2
x_new = np.array([[5.0, 3.4, 1.5, 0.3]])                  # rõ ràng là setosa — chỉ có petal_width = 0.3
for vs in [1e-9, 1e-2]:
    m = GaussianNaiveBayes(var_smoothing=vs).fit(X_const, y_train)
    print(f"var_smoothing = {vs:g}: σ²(setosa, petal_width) = {m.var_[0, 3]:.1e} → dự đoán {m.predict(x_new)[0]:10s}"
          f"| P = {m.predict_proba(x_new).round(3)[0]}")

## 6. Phiên bản tham chiếu

In [ ]:
from ml_course.algorithms import GaussianNBScratch

ref = GaussianNBScratch().fit(X_train, y_train)
print("src/ml_course khớp sklearn:", np.allclose(ref.predict_proba(X_test), sk.predict_proba(X_test)))

## 🧠 Kiểm tra nhanh

1. "Train" Gaussian NB gồm những phép tính nào? Độ phức tạp?
2. Khi nào decision boundary của Gaussian NB là tuyến tính?
3. Gaussian NB có cần scaling feature không? (Gợi ý: thử nhân một feature với 1000 rồi xem $\mu$, $\sigma^2$ và posterior.)
4. Feature có phân phối lệch nặng (thu nhập) — giả định Gaussian có ổn không? Cách xử lý?

## 📌 Tóm tắt

- Gaussian NB: mỗi (lớp, feature) một Gaussian; train = tính mean/var/prior — cực nhanh, $O(nd)$.
- Ranh giới bậc hai (tuyến tính nếu phương sai chung).
- Prior dịch chuyển quyết định; `var_smoothing` chống phương sai 0.
- Không cần scaling (mỗi feature được chuẩn hoá bởi chính $\sigma_{cj}$), nhưng nên biến đổi feature lệch cho gần Gaussian.